# Part 2 — Unbinned likelihood analysis

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import numpy as np

from scipy.optimize import minimize_scalar
from scipy.stats import norm, poisson, truncnorm, uniform, chi2
from tqdm import tqdm

# Settings: font size for plot labels
matplotlib.rcParams['font.size'] = 14

Example in 1D. We consider values of an observable $x$ in the $[-10, 10]$ interval with a uniform background distribution and a Gaussian signal distribution.

In [ ]:
# PDF shapes
xmin, xmax = -10, 10
sigma = 1

# We use rv_continuous classes from scipy.stats, as they provide both the PDF computation (`.pdf(...)`) and the random generation (`.rvs(...)`)
sample_background_pdf = uniform(loc=xmin, scale=xmax-xmin)
sample_signal_pdf = truncnorm(xmin/sigma, xmax/sigma, loc=0, scale=sigma)

In [ ]:
def generate_component(mu, pdf, fluctuate=True):
    # If fluctuate is true, the total number of events is fluctuated using a Poisson distribution. Otherwise, the number of events is fixed.
    n = np.random.poisson(lam=mu) if fluctuate else int(mu)
    # The values of observables are generated using scipy.stats.(...).rvs function.
    return pdf.rvs(size=n)

Generate a dataset with $\mu_s,\mu_b$ the **expected** numbers of signal and background events. The actual numbers of signal and background events have to be randomly sampled from Poisson distributions.

Ignore the `fluctuate_signal` option for now.

In [ ]:
def generate_dataset(mu_signal, mu_background, signal_pdf, background_pdf, fluctuate_signal=True):
    signal = generate_component(mu_signal, signal_pdf, fluctuate=fluctuate_signal)
    background = generate_component(mu_background, background_pdf)
    return np.concatenate((signal, background)), len(signal), len(background)

**Question 1:** Generate a dataset with mu_signal=60 and mu_background=100 and plot the distribution of the variable x.

In [ ]:
mu_signal = 60
mu_background = 100

# Create dataset
data = ...

In [ ]:
# Plot the distribution of observables for the dataset

Log-likelihood function: use an **extended maximum likelihood** method, allowing for Poisson fluctuations of the numbers of signal and background events.

$$\mathcal{L} = e^{-(\mu_s + \mu_b)} \prod_i \mu_s S(x_i) + \mu_b B(x_i)$$

where $S(x), B(x)$ are the signal and background PDFs and $\mu_s,\mu_b$ are the **expected** numbers of signal and background events (which may differ from the observed ones due to Poisson fluctuations). The $x_i$ are the values of the observables for the dataset considered.

In [ ]:
def loglikelihood(data, mu_signal, mu_background, signal_pdf, background_pdf):
    l1 = mu_signal * signal_pdf.pdf(data) + mu_background * background_pdf.pdf(data)
    return np.log(l1).sum() - (mu_signal + mu_background)

**Question 2:** Compute and plot $\lambda(\mu_{\text{sig}}) = 2 \log \dfrac{\mathcal{L}(\hat\mu_{\text{sig}})}{\mathcal{L}(\mu_{\text{sig}})}$, where the hatted value indicates the value maximizing the likelihood.

In [ ]:
# Scan over numbers of signal events
mus = np.arange(0, len(data), 0.1)

# ...

Vary the **true** numbers of signal and background events. Make several trials for each value. 

**QUESTION 3:** Comment on the relative positions of the likelihood optimum and of the true number of signal events

**QUESTION 4:** What happens for low numbers of events? For 0 signal events?

**QUESTION 5:** What does the shape of the log-likelihood near the optimum remind you of? 

### Likelihood fit

In general, we want to be able to extract properly the value of $\mu_{\text{sig}}$ maximising the likelihood. We can then check how often we recover the injected value.

In [ ]:
mu_background = 10

# Fitting function
def likelihood_fit(data, mu_background, signal_pdf, background_pdf):
    if len(data)==0: return 0,0
    l1 = minimize_scalar(lambda mu: -loglikelihood(data, mu, mu_background, signal_pdf, background_pdf), bounds=(0, len(data))) 
    return l1.x, -l1.fun # best-fit expected number of signal events and associated log(L) value

In [ ]:
# Check for bias
def make_bias_plot(mu_min=0, mu_max=10, mu_step=1, mu_background=10, Ntrials=100):
    mus = np.arange(mu_min, mu_max+1, mu_step)
    mus_fit = np.zeros((len(mus), Ntrials))

    for i,mu in enumerate(tqdm(mus)):
         for j in range(Ntrials):
            data, _, _ = generate_dataset(mu, mu_background, sample_signal_pdf, sample_background_pdf)
            mus_fit[i,j] = likelihood_fit(data, mu_background, sample_signal_pdf, sample_background_pdf)[0]

    plt.boxplot(mus_fit.T, '.', positions=mus)
    plt.plot(mus,mus, '--', color='r')
    plt.xlabel("True signal expectation value")
    plt.ylabel("Best-fit signal expectation value")

In [ ]:
make_bias_plot(mu_min=0, mu_max=10, mu_step=1, Ntrials=1000)

### Test statistic

We define the test statistic $\Lambda = 2\log\frac{\mathcal{L}(\hat\mu_{\text{sig}})}{\mathcal{L}(0)}$, where $\hat\mu_s$ is the best-fit value of $\mu_s$.

In [ ]:
# Test statistic
def compute_Lambda(data, mu_background, signal_pdf, background_pdf):
    if len(data) == 0: return 0
    loglike_H1 = likelihood_fit(data, mu_background, signal_pdf, background_pdf)[1]
    return 2*(loglike_H1 - loglikelihood(data, 0, mu_background, signal_pdf, background_pdf))

mu_background = 10
mu_signal = 10

data, _, _ = generate_dataset(mu_signal, mu_background, sample_signal_pdf, sample_background_pdf)
Lambda = compute_Lambda(data, mu_background, sample_signal_pdf, sample_background_pdf)
print("Test statistic for one fake dataset with", mu_signal, "expected signal events is", Lambda)

Build $\Lambda$ distribution for the background hypothesis

In [ ]:
def get_Lambda_distribution(mu_signal, mu_background, signal_pdf, background_pdf, Npseudo_experiments = 10000, fluctuate_signal=True):
    Lambda_table = np.zeros(Npseudo_experiments)
    for i in tqdm(range(Npseudo_experiments)):
        data, _, _ = generate_dataset(mu_signal, mu_background, signal_pdf, background_pdf, fluctuate_signal=fluctuate_signal)
        Lambda_table[i] = compute_Lambda(data, mu_background, signal_pdf, background_pdf)
    return Lambda_table

# For H0, number of pseudo-experiments and background to be simulated
Npseudo_experiments_H0 = 10000
mu_background = 10
# For H1, number of pseudo-experiments and signal to be simulated (background is the same as H0) 
Npseudo_experiments_H1 = 1000
mu_signal = 10

Lambda_H0 = get_Lambda_distribution(0, mu_background, sample_signal_pdf, sample_background_pdf, Npseudo_experiments_H0)
Lambda_H1 = get_Lambda_distribution(mu_signal, mu_background, sample_signal_pdf, sample_background_pdf, Npseudo_experiments_H1)

# H0 distribution
bins = np.linspace(-1, Lambda_H1.max(), 51)
binwidth=bins[1]-bins[0]
plt.hist(Lambda_H0, bins, label=r"$H_0$")

# If the distribution can be approximated by an analytical function, plot it here
# ...

# H1 distributions
plt.hist(Lambda_H1, bins, histtype='step', color='orange', label=r"$H_1$")
plt.xlabel(r"$\Lambda$")
plt.ylabel("Number of pseudo-experiments")
plt.yscale('log')
plt.grid(linestyle=':', which='both')
plt.legend(loc="best")

**QUESTION 6:** For 0 signal events ("null hypothesis" $H_0$), what could the distribution be approximated with?

### Computing p-values

The p-value of a given observation ($\Lambda_{\rm obs}$) is given by the fraction of $H_0$ $\Lambda$ values which are above $\Lambda_{\rm obs}$. These p-values are defined making the correspondence with a Normal distribution: $p_{n\sigma} = \int_{n}^{\infty} \text{Normal}(x) \text{d}x$ (for one-sided p-value).

In [ ]:
one_sided = True
significance_sigma = 2
p_value_threshold = norm.sf(significance_sigma) * (1 if one_sided else 2)

print(f"The p-value for significance of {significance_sigma:d} sigma is {p_value_threshold:.2e}")

In [ ]:
# Compute p-value one generated pseudo-experiment
mu_signal = 3
data, _, _ = generate_dataset(mu_signal, mu_background, sample_signal_pdf, sample_background_pdf)
Lambda_obs = compute_Lambda(data, mu_background, sample_signal_pdf, sample_background_pdf)
nbkg_above_obs = (Lambda_H0 > Lambda_obs).sum()
pvalue = nbkg_above_obs/Npseudo_experiments_H0

# Error can be estimated using binomial error: error(e = n/N) = sqrt(e*(1-e) / N)
err_pvalue = np.sqrt(pvalue*(1-pvalue) / Npseudo_experiments_H0)
print("p-value is", pvalue, f"+- {err_pvalue:.2e}")

### Computing discovery potential

For a given signal hypothesis (value of signal strength), we can compute the median p-value. The $n\sigma$ "discovery potential" of the experiment is defined as the signal strength for which the median p-value is equal to $p_{n\sigma}$. In principle, we should repeat the procedure shown in the cell above for different values of $\mu_{\text{sig}}$, which can be non-integer, to find the discovery potential. This would be very computationally intensive.

Another possible method is to simulate only **fixed** integer numbers of signal events and include Poisson fluctuations later:

$$p_{\mu_{\text{sig}}}(\Lambda) = \sum_i \mathrm{Poisson}(i, \lambda=\mu_{\text{sig}})\; p_{N_s=i}(\Lambda)$$

where $p_{\cdot}(\Lambda)$ indicates the probability distribution of $\Lambda$ under the hypothetical signal indicated as subscript. This is why we added the option `fluctuate_signal` to the dataset generation function. In practice, here, we will build the $\Lambda_{\mu_{\text{sig}})}$ distribution by concatenating subsamples of the $\Lambda$ lists corresponding to integer numbers of signal events. 

In [ ]:
mu_background = 10

# Scan over mu_s
nsigs = np.arange(0, 15)
Npseudo_experiments_H0 = 10000
Npseudo_experiments_H1 = 2000

# Create the arrays of Lambda values for all integer values of number of signal events
Lambda_list = []
for i, nsig in enumerate(nsigs):
    Npseudo_experiments = Npseudo_experiments_H0 if i==0 else Npseudo_experiments_H1
    Lambda_list.append(get_Lambda_distribution(nsig, mu_background, sample_signal_pdf, sample_background_pdf, Npseudo_experiments, fluctuate_signal=False))

**QUESTION 7:** Plot median p-value as a function of mu_signal. Find the value of mu_signal with a median p-value of 2sigma.

In [ ]:
# Get median Lambda for an arbitrary mu value
def median_Lambda(mu_s, Ntoys=1000):
    # Number of events in each subsample
    poisson_distr = poisson.pmf(nsigs, mu=mu_s) * Ntoys # expectation value for each nsig
    numbers = np.random.poisson(lam=poisson_distr)      # apply Poisson fluctuation
    # Get a random subsample of the H0 sample
    indices_H0 = np.arange(0, Npseudo_experiments_H0)
    Lambda_H0 = Lambda_list[0][np.random.permutation(indices_H0)][:numbers[0]]
    # Get random subsample for each number of signal events
    indices_H1 = np.arange(0, Npseudo_experiments_H1)
    Lambda_set = np.concatenate(
                [Lambda_H0] + 
                [
                 ts[np.random.permutation(indices_H1)][:n]
                 for n,ts in zip(numbers[1:], Lambda_list[1:])
                ])
    return np.median(Lambda_set)

xmu = np.arange(0,10,0.01)
medians = np.array([median_Lambda(mu) for mu in xmu])

# Compute p-values
# ...
pvalues = ...

fig, ax = plt.subplots(2, 1, figsize=(6, 6), sharex=True, gridspec_kw={"hspace":0.03})
ax[0].plot(xmu, medians)
ax[0].set_ylabel(r"Median $\Lambda$")

# Plot in ax[1] the p-value as a function of mu_signal
# ...

p_value_threshold = norm.sf(2)
ax[1].axhline(p_value_threshold, color="red", ls="dashed")
ax[1].set_ylabel("p-value")
ax[1].set_yscale("log")
ax[1].set_xlabel("Signal expectation")

### Computing upper limit

Using the same pseudo-experiments, we may also compute sensitivity and upper limits using the distribution $\Lambda$. A comparison of different sensitivity definition is available in the "extra" notebook.

In particular, for a 90% upper limit on $\mu_s$, we estimate it as the minimum value of $\mu_s$ such that 90% of the distribution of $\Lambda$ under this hypothesis $\mu_s$ is above the observed $\Lambda$ value.

In [ ]:
mu_signal = 2
data, _, _ = generate_dataset(mu_signal, mu_background, sample_signal_pdf, sample_background_pdf)
Lambda_obs = compute_Lambda(data, mu_background, sample_signal_pdf, sample_background_pdf)

# Get fraction of the Lambda is above the observed value, for any mu_s
def fraction_above(mu_s, Lambda_obs, Ntoys=1000):
    # Number of events in each subsample
    poisson_distr = poisson.pmf(nsigs, mu=mu_s) * Ntoys # expectation value for each nsig
    numbers = np.random.poisson(lam=poisson_distr)      # apply Poisson fluctuation
    # Get a random subsample of the H0 sample
    indices_H0 = np.arange(0, Npseudo_experiments_H0)
    Lambda_H0 = Lambda_list[0][np.random.permutation(indices_H0)][:numbers[0]]
    # Get random subsample for each number of signal events
    indices_H1 = np.arange(0, Npseudo_experiments_H1)
    Lambda_set = np.concatenate(
                [Lambda_H0] + 
                [
                 ts[np.random.permutation(indices_H1)][:n]
                 for n,ts in zip(numbers[1:], Lambda_list[1:])
                ])
    return np.count_nonzero(Lambda_set[Lambda_set > Lambda_obs]) / len(Lambda_set)

xmu = np.arange(0, 10, 0.01)
fracs = np.array([fraction_above(mu, Lambda_obs) for mu in xmu])

plt.plot(xmu, fracs)
plt.xlabel("Signal expectation")
plt.ylabel(r"Fraction above $\Lambda_\text{obs}$")
plt.axhline(0.90, color="red", ls="dashed")

## EXTRA: Likelihood profile -- Gaussian approximation

Consider again the shape of the log-likelihood as a function of the assumed $\mu_s$. What did it remind you of?

If the true parameter value is **within** the allowed range (**not** on the edge) and there is sufficient statistics, the likelihood near the optimum can be approximated by a Gaussian (second-order Taylor expansion of the log-likelihood):

$$2\log \frac{\mathcal{L}(\mu)}{\mathcal{L}(\mu_\mathrm{best})} \sim -\frac{(\mu - \mu_\mathrm{best})^2}{\sigma_l^2}$$

with $\frac{1}{\sigma_l^2} = \left.\frac{\partial^2 \mathcal{L}}{\partial \mu^2}\right|_{\mu=\mu_\mathrm{best}}$. 

In this case, the $n\sigma$ threshold is defined as a deviation:

$$\lambda = 2\log \dfrac{\mathcal{L}_\mathrm{max}}{\mathcal{L}} = n^2$$

If the true parameter value is on the edge of the allowed range, the first order term of the series expansion can no longer be ignored and the approximation breaks down.

Try it!

In [ ]:
# Vary statistics + true value
mu_signal = 25
mu_background = 50

# Generate random dataset
data, nsig, nbackground = generate_dataset(mu_signal, mu_background, sample_signal_pdf, sample_background_pdf)

if len(data)==0:
    raise RuntimeError("Zero events generated in this dataset. Try again...")

# Scan over numbers of signal events
mus = np.arange(0, len(data), 0.1)
loglike = np.array([loglikelihood(data, mu, mu_background, sample_signal_pdf, sample_background_pdf) for mu in mus])
loglike_max = loglike.max()

print("Best-fit number of signal events:", mus[loglike.argmax()])
print("True number of signal events:", nsig)
print("Expectation value of the number of signal events:", mu_signal)

# Plot -log(likelihood)
plt.plot(mus, 2 * abs(loglike.max()-loglike), label="Exact result")
plt.axvline(nsig, color='r')

plt.ylim(0, 10)
plt.xlim(0, len(data))
plt.grid(which='both')

# Plot parabola
mus_like_min = mus[loglike.argmax()]
second_derivative = np.gradient(np.gradient(loglike, mus), mus)[loglike.argmax()]
std_dev = 1/np.sqrt(-second_derivative)
plt.plot(mus, (mus - mus_like_min)**2/std_dev**2, label="Gaussian approximation")
plt.axhline(1, ls='dashed', color='black')  # 1sigma
plt.axhline(4, ls='dashdot', color='black')  # 2sigma
plt.axhline(9, ls='dotted', color='black')  # 3sigma

plt.xlabel("Expected number of signal events")
plt.ylabel("$-\\log\\dfrac{L}{L_\\mathrm{max}}$")
plt.legend()
plt.show()